In [1]:
# FGBUSTER IMPORTS
import healpy as hp
import pysm3

from fgbuster import (CMB, Dust, Synchrotron, 
                      basic_comp_sep, MixingMatrix,
                      get_observation, get_instrument)
from fgbuster.visualization import corner_norm

# FURAX IMPORTS
import jax
import jaxopt
import jax.numpy as jnp
from jax import ShapeDtypeStruct

from furax._base.blocks import BlockDiagonalOperator, BlockRowOperator
from furax._base.core import HomothetyOperator, IdentityOperator
from furax.landscapes import StokesPyTree, ValidStokesType, HealpixLandscape
from furax.tree import as_structure
from furax.operators.sed import CMBOperator, DustOperator, SynchrotronOperator , MixingMatrixOperator
import operator
from math import prod
import numpy as np
from functools import partial

import os
import pickle

In [9]:
instrument = get_instrument('LiteBIRD')

def generate_maps(nside):
    npixel = nside ** 2 * 12
    # Define cache file path
    cache_dir = 'freq_maps_cache'
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f'freq_maps_nside_{nside}.pkl')

    # Check if file exists, load if it does, otherwise create and save it
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            freq_maps = pickle.load(f)
        print(f"Loaded freq_maps for nside {nside} from cache.")
    else:
        # Generate freq_maps if not already cached
        freq_maps = get_observation(instrument, 'c1d0s0', nside=nside)
        
        # Save freq_maps to the cache
        with open(cache_file, 'wb') as f:
            pickle.dump(freq_maps, f)
        print(f"Generated and saved freq_maps for nside {nside}.")

    # Check the shape of freq_maps
    print("freq_maps shape:", freq_maps.shape)


nsides = [32 , 64 , 128 , 256 , 512]
for nside in nsides:
    generate_maps(nside)

Loaded freq_maps for nside 32 from cache.
freq_maps shape: (15, 3, 12288)
Loaded freq_maps for nside 64 from cache.
freq_maps shape: (15, 3, 49152)
Loaded freq_maps for nside 128 from cache.
freq_maps shape: (15, 3, 196608)
Loaded freq_maps for nside 256 from cache.
freq_maps shape: (15, 3, 786432)
Loaded freq_maps for nside 512 from cache.
freq_maps shape: (15, 3, 3145728)


In [10]:
nside = 32
npixel = nside ** 2 * 12
dust_nu0 = 150.0
synchrotron_nu0 = 20.0
stokes_type: ValidStokesType = 'IQU'
instrument = get_instrument('LiteBIRD')

# Define cache file path
cache_dir = 'freq_maps_cache'
cache_file = os.path.join(cache_dir, f'freq_maps_nside_{nside}.pkl')

# Check if file exists and load if it does; otherwise raise an error with guidance
if os.path.exists(cache_file):
    with open(cache_file, 'rb') as f:
        freq_maps = pickle.load(f)
    print(f"Loaded freq_maps for nside {nside} from cache.")
else:
    raise FileNotFoundError(
        f"Cache file for freq_maps with nside {nside} not found.\n"
        f"Please generate it first by calling `generate_maps({nside})`."
    )

# Check the shape of freq_maps
print("freq_maps shape:", freq_maps.shape)

Loaded freq_maps for nside 32 from cache.
freq_maps shape: (15, 3, 12288)


In [15]:
d = StokesPyTree.from_stokes(Q=freq_maps[:,1,:], U=freq_maps[:,2,:])
N = HomothetyOperator(jnp.ones(1), _in_structure=d.structure)
nu = instrument['frequency'].values

In [16]:
from furax.comp_sep import spectral_cmb_variance

spectral_cmb_variance = partial(spectral_cmb_variance , dust_nu0=dust_nu0, synchrotron_nu0=synchrotron_nu0)

In [17]:
from furax.comp_sep import optimize
import optax
import optax.tree_utils as otu

solver = optax.lbfgs()

params = {"beta_dust": 1.59, "beta_pl": -3.1, "temp_dust": 19.6}

final_params, final_state = optimize(params,
                                     spectral_cmb_variance,
                                     solver,
                                     max_iter=100,
                                     tol=1e-4,
                                     nu=nu,
                                     N=N,
                                     d=d)

print(
    f"Final parameters: {final_params}, number of evaluations: {otu.tree_get(final_state, 'count')}"
)
print(f"Initial Value: {spectral_cmb_variance(final_params , nu=nu, N=N, d=d)}")


Final parameters: {'beta_dust': Array(1.54888217, dtype=float64), 'beta_pl': Array(-3.05748183, dtype=float64), 'temp_dust': Array(19.59906649, dtype=float64)}, number of evaluations: 12
Initial Value: 0.3021739117954524


In [26]:
from furax.comp_sep import DistributedGridSearch

def objective_function(beta_dust , temp_dust , beta_pl):

    params = {
        "beta_dust": beta_dust,
        "temp_dust": temp_dust,
        "beta_pl": beta_pl,
    }

    return spectral_cmb_variance(params , nu=nu , N=N , d=d)


# Put the good values for the grid search
search_space = {
    "beta_dust": jnp.linspace(1.5, 3.5, 10).at[3].set(final_params["beta_dust"]),
    "temp_dust": jnp.linspace(5., 50., 10).at[2].set(final_params["temp_dust"]),
    "beta_pl": jnp.linspace(-4.5, -1.5, 10).at[4].set(final_params["beta_pl"]),
}

grid_search = DistributedGridSearch(objective_function, search_space, batch_size=25, progress_bar=True ,log_every=0.1)

results = grid_search.run()

for keys, values in results.items():
    print(f"{keys} : {values[0]}")

Selecting batch size of 25
log_interval: 4


Processing batches: 100%|██████████| 40/40 [00:05<00:00,  7.68it/s]


Done .. Stacking the results
beta_dust : 1.5488821707719607
temp_dust : 19.59906649076405
beta_pl : -3.0574818308646994
value : 0.30217391179544223
